In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. 定義分類、合併規則與統一配色
# ==========================================

# 定義統一的年份顏色對照表 (確保兩張圖顏色一致)
year_colors = {
    '2021-2022': '#ff7f0e',  # 橘色
    '2023-2024': '#1f77b4',  # 藍色
    '2025-2026': '#2ca02c'  # 綠色
}

# 圖表一：廣泛概念詞彙
concept_rules = {
    'ai': 'ai',
    '人工智慧': 'ai',
    '模型': 'model',
    'model': 'model',
    'deep': 'deep learning',
    'deeplearning': 'deep learning',
    'machine': 'machine learning',
    'learning': 'machine learning',
    'ml': 'machine learning',
    'llm': 'llm',
    'token': 'token',
    'agent': 'agent',
    'dl': 'dl',
    'rl': 'rl',
    'nlp': 'nlp',
    'prompt': 'prompt',
    'mle': 'mle',
    'reinforcement': 'rl',
    'neural': 'neural',
    'vae': 'vae'
}

# 圖表二：實體工具與企業
tool_rules = {
    'chatgpt': 'chatgpt',
    'gpt': 'chatgpt',
    'gemini': 'gemini',
    'cursor': 'cursor',
    'copilot': 'copilot',
    'claude': 'claude',
    'llama': 'llama',
    'openai': 'openai',
    'deepmind': 'deepmind',
    'codewhisperer': 'codewhisperer',
    'alphago': 'alphago',
    'alphafold': 'alphafold'
}

files = {
    '2021-2022': '2021-2022詞頻計算最終版.csv',
    '2023-2024': '2023-2024詞頻計算最終版.csv',
    '2025-2026': '2025-2026詞頻計算最終版.csv'
}

all_concept_data = []
all_tool_data = []

# ==========================================
# 2. 讀取資料並套用分類規則
# ==========================================
for year, file_path in files.items():
    try:
        df = pd.read_csv(file_path)
        if 'Term' in df.columns and 'n' in df.columns:
            # 確保轉為小寫
            df['Term_lower'] = df['Term'].astype(str).str.lower()

            # --- 處理圖表一 (廣泛概念) ---
            concept_df = df[df['Term_lower'].isin(concept_rules.keys())].copy()
            concept_df['Term_Mapped'] = concept_df['Term_lower'].map(concept_rules)
            concept_df['Year'] = year
            all_concept_data.append(concept_df)

            # --- 處理圖表二 (實體工具) ---
            tool_df = df[df['Term_lower'].isin(tool_rules.keys())].copy()
            tool_df['Term_Mapped'] = tool_df['Term_lower'].map(tool_rules)
            tool_df['Year'] = year
            all_tool_data.append(tool_df)

    except FileNotFoundError:
        print(f"找不到檔案：{file_path}，請確認路徑。")


# ==========================================
# 3. 定義繪圖函數 (加入顏色對照表)
# ==========================================
def plot_trend(data_list, title, filename):
    if not data_list:
        return

    # 合併該組的所有年份資料
    combined_df = pd.concat(data_list, ignore_index=True)

    # 將轉換過相同名稱的詞彙頻率加總
    grouped_df = combined_df.groupby(['Term_Mapped', 'Year'])['n'].sum().reset_index()

    # 抓出總聲量前 15 名
    top_terms = grouped_df.groupby('Term_Mapped')['n'].sum().sort_values(ascending=False).head(15).index
    plot_df = grouped_df[grouped_df['Term_Mapped'].isin(top_terms)]

    # 開始繪圖
    plt.figure(figsize=(15, 8))

    # 【關鍵修改】在這裡加上 palette=year_colors，強制套用我們設定的顏色字典
    ax = sns.barplot(data=plot_df, x='Term_Mapped', y='n', hue='Year',
                     palette=year_colors, edgecolor='black', order=top_terms)

    # 加上數字標籤
    for container in ax.containers:
        ax.bar_label(container, padding=3, fontsize=10, color='black')

    plt.title(title, fontsize=18, fontweight='bold', pad=20)
    plt.xlabel('分析詞彙', fontsize=14, fontweight='bold')
    plt.ylabel('出現次數 (n)', fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right', fontsize=13)
    plt.legend(title='時間區段', title_fontsize='14', fontsize='12', loc='upper right')

    plt.tight_layout()

    # 自動存檔
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()


# ==========================================
# 4. 執行繪圖輸出
# ==========================================
# 設定 Windows 支援的中文字體，避免亂碼
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei']
plt.rcParams['axes.unicode_minus'] = False

print("開始繪製圖表一：廣泛概念趨勢...")
plot_trend(all_concept_data, '2021-2026 年 PTT 軟體版【廣泛 AI 概念】討論趨勢', 'ai_concepts_trend.png')

print("開始繪製圖表二：實體工具與企業趨勢...")
plot_trend(all_tool_data, '2021-2026 年 PTT 軟體版【AI 實體工具/企業】討論趨勢', 'ai_tools_trend.png')

print("✅ 兩張圖表皆已產生，並儲存至專案資料夾中！")